# 04. Huấn luyện nhóm mô hình tuyến tính
Đọc kết quả từ notebook 3
- Chon dataset variant tốt nhất cho linear
- Chạy từng model linear/non-tree và lưu kết quả.


In [1]:
import sys
import json
import time
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == 'research' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import joblib
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.linear_model import LinearRegression, Ridge, SGDRegressor
from sklearn.model_selection import cross_val_predict
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import LinearSVR, SVR

from src.data.split_data import attach_target, split_features_target
from src.models.evaluate import build_kfold, evaluate_regression_model, evaluate_regression_metrics


## 1.Khai báo đường dẫn và helper


In [2]:
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
METRICS_DIR = PROJECT_ROOT / 'outputs' / 'metrics'
MODELS_DIR = PROJECT_ROOT / 'outputs' / 'models' / 'linear_models'

METRICS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COL = 'Rings'
RANDOM_STATE = 42

def save_table(df, output_stem):
    df.to_csv(output_stem.with_suffix('.csv'), index=False)
    output_stem.with_suffix('.json').write_text(
        json.dumps(df.to_dict(orient='records'), indent=2),
        encoding='utf-8',
    )

def load_processed_datasets():
    split_files = {
        'train': {
            'encoded_only': PROCESSED_DIR / 'abalone_train_encoded.csv',
            'standard_scaled': PROCESSED_DIR / 'abalone_train_standard.csv',
            'robust_log_scaled': PROCESSED_DIR / 'abalone_train_robust_log.csv',
        },
        'test': {
            'encoded_only': PROCESSED_DIR / 'abalone_test_encoded.csv',
            'standard_scaled': PROCESSED_DIR / 'abalone_test_standard.csv',
            'robust_log_scaled': PROCESSED_DIR / 'abalone_test_robust_log.csv',
        },
    }
    has_explicit_splits = all(path.exists() for version in split_files.values() for path in version.values())
    datasets = {'train': {}, 'test': {}}
    if has_explicit_splits:
        for split_name, version_map in split_files.items():
            for version_name, csv_path in version_map.items():
                df = pd.read_csv(csv_path)
                datasets[split_name][version_name] = (df.drop(columns=[TARGET_COL]), df[TARGET_COL])
        return datasets

    encoded_df = pd.read_csv(PROCESSED_DIR / 'abalone_processed_no_scaling.csv')
    scaled_df = pd.read_csv(PROCESSED_DIR / 'abalone_processed_scaled.csv')
    x_train_encoded, x_test_encoded, y_train_encoded, y_test_encoded = split_features_target(encoded_df, TARGET_COL, test_size=0.3, random_state=RANDOM_STATE)
    x_train_scaled, x_test_scaled, y_train_scaled, y_test_scaled = split_features_target(scaled_df, TARGET_COL, test_size=0.3, random_state=RANDOM_STATE)
    train_encoded_df = attach_target(x_train_encoded, y_train_encoded, TARGET_COL)
    test_encoded_df = attach_target(x_test_encoded, y_test_encoded, TARGET_COL)
    train_scaled_df = attach_target(x_train_scaled, y_train_scaled, TARGET_COL)
    test_scaled_df = attach_target(x_test_scaled, y_test_scaled, TARGET_COL)
    datasets['train']['encoded_only'] = (train_encoded_df.drop(columns=[TARGET_COL]), train_encoded_df[TARGET_COL])
    datasets['test']['encoded_only'] = (test_encoded_df.drop(columns=[TARGET_COL]), test_encoded_df[TARGET_COL])
    datasets['train']['standard_scaled'] = (train_scaled_df.drop(columns=[TARGET_COL]), train_scaled_df[TARGET_COL])
    datasets['test']['standard_scaled'] = (test_scaled_df.drop(columns=[TARGET_COL]), test_scaled_df[TARGET_COL])
    datasets['train']['robust_log_scaled'] = (train_scaled_df.drop(columns=[TARGET_COL]), train_scaled_df[TARGET_COL])
    datasets['test']['robust_log_scaled'] = (test_scaled_df.drop(columns=[TARGET_COL]), test_scaled_df[TARGET_COL])
    return datasets

def run_cv_metrics(model, x_data, y_data):
    cv = build_kfold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    y_oof_pred = cross_val_predict(model, x_data, y_data, cv=cv, n_jobs=1)
    return evaluate_regression_metrics(y_data, y_oof_pred)

def evaluate_model(model, x_train, y_train, x_test, y_test):
    cv_start = time.perf_counter()
    cv_metrics = run_cv_metrics(clone(model), x_train, y_train)
    cv_time_sec = time.perf_counter() - cv_start
    fitted_model = clone(model)
    fit_start = time.perf_counter()
    fitted_model.fit(x_train, y_train)
    fit_time_sec = time.perf_counter() - fit_start
    y_pred = fitted_model.predict(x_test)
    test_metrics = evaluate_regression_model(y_test, y_pred)
    return fitted_model, cv_metrics, test_metrics, cv_time_sec, fit_time_sec


## 2. Chọn phiên bản dữ liệu cho nhóm tuyến tính


In [3]:
feature_summary = pd.read_csv(METRICS_DIR / 'feature_selection_baseline_summary.csv')
best_linear_variant = (
    feature_summary.query("family == 'linear_probe'")
    .sort_values(['mean_test_rmse', 'mean_test_mae'])
    .iloc[0]['dataset_variant']
)
print(f'Best linear dataset variant: {best_linear_variant}')


Best linear dataset variant: robust_log_scaled


## 3. Nạp dữ liệu và khai báo các mô hình tuyến tính


In [4]:
datasets = load_processed_datasets()
x_train, y_train = datasets['train'][best_linear_variant]
x_test, y_test = datasets['test'][best_linear_variant]

linear_models = {
    'LinearRegression': LinearRegression(),
    'KNeighborsRegressor': KNeighborsRegressor(),
    'RidgeRegression': Ridge(),
    'SGDRegressor': SGDRegressor(random_state=RANDOM_STATE, max_iter=2000, tol=1e-3),
    'SVR': SVR(),
    'LinearSVR': LinearSVR(random_state=RANDOM_STATE, max_iter=10000),
    'MLPRegressor': MLPRegressor(random_state=RANDOM_STATE, max_iter=1000),
}
print(f'So model se chay: {len(linear_models)}')


So model se chay: 7


## 4. Chạy nhóm mô hình tuyến tính và non-tree


In [5]:
records = []
for model_name, model in linear_models.items():
    fitted_model, cv_metrics, test_metrics, cv_time_sec, fit_time_sec = evaluate_model(
        model, x_train, y_train, x_test, y_test
    )
    joblib.dump(fitted_model, MODELS_DIR / f'{model_name}.joblib')
    records.append({
        'group_name': 'linear_models',
        'model_name': model_name,
        'dataset_variant': best_linear_variant,
        'cv_rmse': cv_metrics['rmse'],
        'cv_mae': cv_metrics['mae'],
        'cv_rse': cv_metrics['rse'],
        'test_rmse': test_metrics['rmse'],
        'test_mae': test_metrics['mae'],
        'test_rse': test_metrics['rse'],
        'test_r2': test_metrics['r2'],
        'cv_time_sec': cv_time_sec,
        'fit_time_sec': fit_time_sec,
    })

linear_results = pd.DataFrame(records).sort_values(['test_rmse', 'test_mae', 'cv_rmse']).reset_index(drop=True)
linear_results['rank_in_group'] = range(1, len(linear_results) + 1)
display(linear_results)


,group_name,model_name,dataset_variant,cv_rmse,cv_mae,cv_rse,test_rmse,test_mae,test_rse,test_r2,cv_time_sec,fit_time_sec,rank_in_group
0,linear_models,MLPRegressor,robust_log_scaled,2.173823,1.499122,0.450625,2.109222,1.503208,0.438110,0.561890,18.114501,3.299607,1
1,linear_models,RidgeRegression,robust_log_scaled,2.188713,1.566469,0.456819,2.166889,1.557175,0.462394,0.537606,0.025312,0.003068,2
2,linear_models,SVR,robust_log_scaled,2.197660,1.490488,0.460561,2.174461,1.495234,0.465632,0.534368,1.894501,0.513135,3
3,linear_models,LinearSVR,robust_log_scaled,2.243512,1.543189,0.479980,2.175788,1.517793,0.466200,0.533800,0.072255,0.013653,4
4,linear_models,LinearRegression,robust_log_scaled,2.184263,1.568446,0.454963,2.177008,1.559670,0.466723,0.533277,0.086999,0.002912,5
5,linear_models,SGDRegressor,robust_log_scaled,2.247976,1.587754,0.481892,2.177165,1.582212,0.466790,0.533210,0.092925,0.015955,6
6,linear_models,KNeighborsRegressor,robust_log_scaled,2.345189,1.641875,0.524472,2.269786,1.595375,0.507351,0.492649,0.077122,0.008139,7


## 5. Top mô hình tuyến tính cơ sở


In [6]:
display(linear_results.nsmallest(5, 'test_rmse')[['model_name', 'dataset_variant', 'test_rmse', 'test_mae', 'test_r2']])


,model_name,dataset_variant,test_rmse,test_mae,test_r2
0,MLPRegressor,robust_log_scaled,2.109222,1.503208,0.561890
1,RidgeRegression,robust_log_scaled,2.166889,1.557175,0.537606
2,SVR,robust_log_scaled,2.174461,1.495234,0.534368
3,LinearSVR,robust_log_scaled,2.175788,1.517793,0.533800
4,LinearRegression,robust_log_scaled,2.177008,1.559670,0.533277


## 6. Lưu kết quả


In [7]:
save_table(linear_results, METRICS_DIR / 'linear_models_results')
print('Đã lưu kết quả linear models vào outputs/metrics/ và outputs/models/linear_models/.')


Đã lưu kết quả linear models vào outputs/metrics/ và outputs/models/linear_models/.
